# Proyecto HR

In [3]:
import pandas as pd
import numpy as np
import mysql.connector
from datetime import datetime

# --- 1. GENERACIÓN DE DATOS DIMENSIONALES ---
depts_data = {
    'dept_id': [1, 2, 3, 4],
    'nombre_dept': ['Ventas', 'Tecnología', 'Recursos Humanos', 'Marketing'],
    'sueldo_base': [35000, 55000, 32000, 38000]
}
df_depts = pd.DataFrame(depts_data)

# --- 2. GENERACIÓN DE EMPLEADOS (CON DATOS "SUCIOS") ---
n_empleados = 500
nombres_sucios = ['  JUAN perez', 'maria Garcia  ', 'luis T.', 'ana r.', 'PEDRO s.'] * 100
fechas_sucias = ['2022-01-10', '15/05/2021', '2023.08.01', '2020-10-25', '01-12-2022'] * 100

df_emp = pd.DataFrame({
    'emp_id': range(1, n_empleados + 1),
    'nombre': nombres_sucios,
    'genero': np.random.choice(['Masculino', 'Femenino'], n_empleados),
    'fecha_contratacion': fechas_sucias
})

# --- 3. PROCESO DE LIMPIEZA (DATA CLEANING) ---
# A. Limpieza de texto: Quitar espacios extras y formato Título (Juan Perez)
df_emp['nombre'] = df_emp['nombre'].str.strip().str.title()

# B. Estandarización de fechas: Convertir formatos mixtos a YYYY-MM-DD
df_emp['fecha_contratacion'] = pd.to_datetime(df_emp['fecha_contratacion'], dayfirst=False, errors='coerce').dt.date

# --- 4. GENERACIÓN DE HECHOS (DESEMPEÑO) ---
df_fact = pd.DataFrame({
    'eval_id': range(1, n_empleados + 1),
    'emp_id': range(1, n_empleados + 1),
    'dept_id': np.random.randint(1, 5, n_empleados),
    'puntaje_eval': np.random.randint(1, 6, n_empleados), # Escala 1 a 5
    'dias_ausente': np.random.randint(0, 15, n_empleados),
    'estatus_activo': np.random.choice(['Si', 'No'], n_empleados, p=[0.85, 0.15]) # 15% de rotación
})

# --- 5. CARGA A MYSQL ---
try:
    conn = mysql.connector.connect(
        host="localhost",
        user="root",
        password="xxxxxxx", # <--- Pass
        database="ProyectoHR"
    )
    cursor = conn.cursor()

    # Cargar Departamentos
    for i, row in df_depts.iterrows():
        cursor.execute("INSERT INTO Dim_Departamentos VALUES (%s, %s, %s)", tuple(row))
    
    # Cargar Empleados (Limpios)
    for i, row in df_emp.iterrows():
        cursor.execute("INSERT INTO Dim_Empleados VALUES (%s, %s, %s, %s)", tuple(row))
        
    # Cargar Desempeño
    for i, row in df_fact.iterrows():
        cursor.execute("INSERT INTO Fact_Desempeno VALUES (%s, %s, %s, %s, %s, %s)", tuple(row))

    conn.commit()
    print(f"✅ ¡Proceso completado! 500 registros limpios cargados en ProyectoHR.")

except mysql.connector.Error as err:
    print(f"❌ Error: {err}")
finally:
    if conn.is_connected():
        conn.close()

✅ ¡Proceso completado! 500 registros limpios cargados en ProyectoHR.
